### Math Foundations

**1. Matrix multiplication** Matrix is a grid of numbers, shape (rows, columns). Matrix multiplication C = A @ B: entry C[i,j] is the dot product of A's row i and B's column j. Requires A's column count to equal B's row count. Every entry is one dot product. 

```
X = [[1, 0, 1],      W = [[2, 0, 1],     W^T = [[2, 1],
     [0, 1, 1]]           [1, 1, 0]]            [0, 1],
                                                  [1, 0]]

Z[0,0] = X_row0 . W^T_col0 = [1,0,1].[2,0,1] = 1*2+0*0+1*1 = 3
Z[0,1] = X_row0 . W^T_col1 = [1,0,1].[1,1,0] = 1*1+0*1+1*0 = 1
Z[1,0] = X_row1 . W^T_col0 = [0,1,1].[2,0,1] = 0*2+1*0+1*1 = 1
Z[1,1] = X_row1 . W^T_col1 = [0,1,1].[1,1,0] = 0*1+1*1+1*0 = 1

Z = [[3, 1],
     [1, 1]]
```
**2. Transpose and identity** Transpose (A^T): flip rows and columns, A[i,j] becomes A^T[j,i]. To make the inner dimensions match). Identity matrix (I): the matrix equivalent of the number 1, ones on the diagonal, zeros elsewhere, A @ I = A for any compatible A. 

**3. Matrix inverse** A^-1 such that A @ A^-1 = I. For a 2x2 matrix [[a,b],[c,d]], the formula is direct: A^-1 = (1/det(A)) * [[d,-b],[-c,a]], where det(A) = ad-bc.

A = [[4, 2], [2, 3]]:
```
det(A) = 4*3 - 2*2 = 12-4 = 8
A^-1 = (1/8) * [[3, -2], [-2, 4]] = [[0.375, -0.25], [-0.25, 0.5]]
```
This is exactly the operation `np.linalg.inv(X_design.T @ X_design)` performs inside linear regression's normal equation `w = (X^T X)^-1 X^T y`, solving for the weights directly without any iterative gradient descent. If det(A)=0, the matrix has NO inverse (singular), which is exactly what happens under severe multicollinearity (near-duplicate features), flagged as an assumption violation Linear Regression section.

**4. Eigenvalues and eigenvectors**

For a square matrix A, an eigenvector v and eigenvalue lambda satisfy A @ v = lambda * v, applying A to v just scales it, does not change its direction. PCA's entire mechanism (finding the directions of maximum variance) is exactly finding the eigenvectors of the covariance matrix, the eigenvector with the largest eigenvalue is the direction PC1 points.

**5. Vector norms**

L2 norm: ||v||_2 = sqrt(sum(v_i^2)), the ordinary (Euclidean) length of a vector. Ridge's penalty is lambda times the squared L2 norm of the weight vector.

L1 norm: ||v||_1 = sum(|v_i|), the sum of absolute values. Lasso's penalty term, lambda times the L1 norm of the weight vector.

The geometric difference between these two norms (L2's circular "equal distance" contour vs. L1's diamond-shaped contour) is the actual geometric reason Lasso can produce exact zeros and Ridge cannot, the diamond shape's corners land exactly on the axes, where one coordinate is exactly zero.

In [ ]:
import numpy as np

a = np.array([1, 2])
b = np.array([3, 4])

print("addition:", a + b)
print("scalar multiply:", 2 * a)
print("dot product:", np.dot(a, b), "(or a @ b:", a @ b, ")")

X = np.array([[1, 0, 1], [0, 1, 1]])
W = np.array([[2, 0, 1], [1, 1, 0]])

Z = X @ W.T
print("Z:\n", Z)
print("shapes: X", X.shape, "@ W.T", W.T.shape, "->", Z.shape)

A = np.array([[4, 2], [2, 3]])
det_A = np.linalg.det(A)
A_inv = np.linalg.inv(A)

print("det(A):", det_A)
print("A inverse:\n", A_inv)
print("check, A @ A_inv (should be identity):\n", np.round(A @ A_inv, 6))

#### Calculus and Differentiation

**1. Derivatives: rate of change**

Definition: f'(x) = lim(h->0) [f(x+h) - f(x)] / h, the instantaneous rate of change of f at x, the slope of the tangent line at that exact point.

f(x) = x^2, at x=3, approximated with a small but nonzero h=0.001 (this is literally what the limit definition converges to as h shrinks toward 0):
```
f(3) = 9
f(3.001) = 3.001^2 = 9.006001

f'(3) ~ (9.006001 - 9) / 0.001 = 0.006001/0.001 = 6.001
```

**2. Partial derivatives**

For a function of MULTIPLE variables, the partial derivative with respect to one variable holds every other variable fixed and measures the rate of change along just that one direction.

f(w,x) = w*x + w^2 , at w=2, x=3:
```
df/dw = x + 2w    (treating x as constant)   = 3 + 2*2 = 7
df/dx = w         (treating w as constant)   = 2
```
"if I nudge only this one parameter, holding everything else fixed, how much does the output change."

**3. Chain rule**

Formula: if y = f(g(x)), then dy/dx = df/dg * dg/dx, differentiate the outer function, then multiply by the derivative of the inner function. Ex - logreg derivation step by step: L depends on p, p depends on z, z depends on w.
```
dL/dw = dL/dp * dp/dz * dz/dw
```
```
z = 1.1, p = sigmoid(1.1) = 0.7503

dL/dp = (p-y)/(p*(1-p))   [derivative of -[y*ln(p)+(1-y)*ln(1-p)] w.r.t. p]
      = (0.7503-1)/(0.7503*0.2497) = -0.2497/0.1873 = -1.333

dp/dz = p*(1-p) = 0.7503*0.2497 = 0.1873          [sigmoid's own derivative]

dz/dw = x = 2                                       [z=w*x+b, so dz/dw=x]

dL/dw = (-1.333) * (0.1873) * (2) = -0.4994
```
The (p-y)/(p*(1-p)) and p*(1-p) terms cancel exactly, leaving(p-y)*x 

**1. Softmax**

Softmax is a mathematical function that converts a vector of raw, unnormalized numbers (called logits) into a probability distribution.

It is most commonly used as the final activation layer in multi-class classification models and Transformer architectures (such as attention mechanisms).
Formula : Given a vector of $N$ raw scores $z = [z_1, z_2, \dots, z_N]$, the Softmax output $P_i$ for each element $i$ is calculated as:$$\text{Softmax}(z_i) = \frac{e^{z_i}}{\sum_{j=1}^{N} e^{z_j}}$$

Key Steps:
- Exponentiate ($e^{z_i}$): Raises Euler's number $e$ to the power of each input score. This turns all negative numbers positive and amplifies larger scores.
- Sum ($\sum e^{z_j}$): Adds all exponential values together to get a total denominator
- Normalize: Divides each exponential value by the total sum, forcing all output values to fall between 0 and 1, with their sum equaling 1.0.